In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import threading
import time
import sys
sys.path.insert(0, '/home/xilinx')
from angus_regs import AngusRegs

STATES = {0:'UNSYNC', 1:'FIRST_GAP', 2:'SYNC_CRANK', 3:'SYNC_FULL'}

# --- Parameter widgets ---
w_kp    = widgets.BoundedIntText(value=256,  min=0, max=65535, description='KP:')
w_ki    = widgets.BoundedIntText(value=16,   min=0, max=65535, description='KI:')
w_maxc  = widgets.BoundedIntText(value=1024, min=0, max=65535, description='Max corr:')
w_phase = widgets.BoundedIntText(value=1717, min=0, max=7199,  description='Phase ang:')
w_tol   = widgets.BoundedIntText(value=300,  min=0, max=7199,  description='Phase tol:')
w_corr_dir = widgets.ToggleButton(value=False, description='Corr dir: SUB', button_style='info')
w_write = widgets.Button(description='Write all', button_style='primary')

# --- Status widgets (left column) ---
w_state   = widgets.Label(value='sync: ---')
w_synced  = widgets.Label(value='synced: ---')
w_sig     = widgets.Label(value='signal_present: ---')
w_rpm     = widgets.Label(value='RPM: ---')
w_losses  = widgets.Label(value='sync losses: ---')
w_pkt     = widgets.Label(value='pkt_count: ---')
w_ovf     = widgets.Label(value='ovf_count: ---')

# --- Debug widgets (right column) ---
w_ph_err  = widgets.Label(value='phase err: ---')
w_corr    = widgets.Label(value='correction: ---')
w_ab      = widgets.Label(value='ab_count: ---')
w_tooth   = widgets.Label(value='tooth_per: ---')
w_gap     = widgets.Label(value='gap_per: ---')
w_nco     = widgets.Label(value='nco_inc: ---')
w_cam     = widgets.Label(value='cam_angle: ---')
w_raw     = widgets.Label(value='raw_angle: ---')
w_crank   = widgets.Label(value='crank_angle: ---')
w_eng     = widgets.Label(value='engine_angle: ---')

err_history  = []
corr_history = []
running = False

def sign32(v):
    return v - 0x100000000 if v > 0x7FFFFFFF else v

def write_all(b):
    regs._raw_write(0x08, w_kp.value)
    regs._raw_write(0x0C, w_ki.value)
    regs._raw_write(0x10, w_maxc.value)
    regs._raw_write(0x14, w_phase.value)
    regs._raw_write(0x18, w_tol.value)
    ctrl = regs._raw_read(0x00)
    if w_corr_dir.value:
        ctrl = ctrl | 0x04
        w_corr_dir.description = 'Corr dir: ADD'
    else:
        ctrl = ctrl & ~0x04
        w_corr_dir.description = 'Corr dir: SUB'
    regs._raw_write(0x00, ctrl)
    print(f'Written: KP={w_kp.value} KI={w_ki.value} MAXC={w_maxc.value} '
          f'PHASE={w_phase.value} TOL={w_tol.value} DIR={"ADD" if w_corr_dir.value else "SUB"}')

w_write.on_click(write_all)

def poll():
    while running:
        try:
            status = regs._raw_read(0x24)
            tp     = regs._raw_read(0x48)
            pe     = sign32(regs._raw_read(0x54))
            co     = sign32(regs._raw_read(0x58))
            sl     = regs._raw_read(0x28)
            ab     = regs._raw_read(0x44)
            gap    = regs._raw_read(0x4C)
            cam    = regs._raw_read(0x5C)
            nco    = regs._raw_read(0x50)
            raw    = regs._raw_read(0x38)
            crank  = regs._raw_read(0x3C)
            eng    = regs._raw_read(0x40)
            pkt    = regs._raw_read(0x30)
            ovf    = regs._raw_read(0x34)

            sync_st = status & 0x7
            synced  = (status >> 5) & 0x1
            sig_pr  = (status >> 4) & 0x1
            ph_flt  = (status >> 3) & 0x1

            rpm    = round(60_000_000 / (tp/100) / 60) if tp > 0 else 0
            ph_deg = pe / 4_294_967_296 * 360

            w_state.value  = f'sync: {STATES.get(sync_st, "?")}  phase_fault={ph_flt}'
            w_synced.value = f'synced: {synced}'
            w_sig.value    = f'signal_present: {sig_pr}'
            w_rpm.value    = f'RPM: {rpm}'
            w_losses.value = f'sync losses: {sl}'
            w_pkt.value    = f'pkt_count: {pkt}'
            w_ovf.value    = f'ovf_count: {ovf}'

            w_ph_err.value = f'phase err: {ph_deg:.3f} deg  ({pe})'
            w_corr.value   = f'correction: {co}'
            w_ab.value     = f'ab_count: {ab}  (of 60)'
            w_tooth.value  = f'tooth_per: {tp}  ({tp/100000:.2f}ms)'
            w_gap.value    = f'gap_per: {gap}  ({gap/100000:.1f}ms)'
            w_nco.value    = f'nco_inc: {nco}  (expected ~71)'
            w_cam.value    = f'cam_angle: {cam}  ({cam/10:.1f} deg)'
            w_raw.value    = f'raw_angle: {raw}  ({raw/10:.1f} deg)'
            w_crank.value  = f'crank_angle: {crank}  ({crank/10:.1f} deg)'
            w_eng.value    = f'engine_angle: {eng}  ({eng/10:.1f} deg)'

            err_history.append(ph_deg)
            corr_history.append(float(co))
            if len(err_history) > 120:
                err_history.pop(0)
                corr_history.pop(0)

        except Exception as e:
            w_state.value = f'error: {e}'
        time.sleep(0.1)

# --- Layout ---
params_box = widgets.VBox([
    widgets.HTML('<b>PLL Parameters</b>'),
    w_kp, w_ki, w_maxc, w_phase, w_tol, w_corr_dir, w_write
])
status_box = widgets.VBox([
    widgets.HTML('<b>Status</b>'),
    w_state, w_synced, w_sig, w_rpm, w_losses, w_pkt, w_ovf
])
debug_box = widgets.VBox([
    widgets.HTML('<b>Debug Registers</b>'),
    w_ph_err, w_corr, w_ab, w_tooth, w_gap, w_nco, w_cam,
    w_raw, w_crank, w_eng
])

display(widgets.HBox([params_box, status_box, debug_box]))

running = True
t = threading.Thread(target=poll, daemon=True)
t.start()
print('Polling started')

In [ ]:
# Plot phase error and correction history - run to refresh
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
ax1.plot(err_history, color='steelblue', linewidth=1.5)
ax1.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax1.set_ylabel('Phase err (deg)')
ax1.set_title('PLL phase error')
ax1.grid(True, alpha=0.3)
ax2.plot(corr_history, color='coral', linewidth=1.5)
ax2.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax2.set_ylabel('Correction (LSB)')
ax2.set_xlabel('Sample (0.1s each)')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
running = False
print('Stopped')